we have 3 files to use- adt raw csv
fate readout csv at barcode lvl 
x rna counts mtx 

subset to hsc . find hvgs . 
use the fate readout csv to classify in terms of fate what barcode lineages are unipotent 
broadcast those lineages onto every cell in the barcode lineage
this is the basis of our groups = True in mofa. We will run with some larger groups first- ery / Mast [largest presence iirc] / cDC1 VS multipotent . see wht makes these JUST ery and not multipotent. so run w groups=True for group= i for all i in celltypes . 

to check if this has worked, check the variance explained per factor in the data. 

if it doesnt work: try run on all data and not just hscs 

In [1]:
import pandas as pd
import scanpy as sc 
import numpy as np 

In [2]:
adt_counts = pd.read_csv("/vast/projects/Sisseq/givanna/making_sense_of_files/adt_raw.csv")

print(adt_counts.head())
print(adt_counts.columns.to_list())

# important here -> protein is column , cell is row so (cell x protien)



   Unnamed: 0                cell  Hu.CD134  Hu.CD112  Hu.HLA.DR  Hu.CD115  \
0           1  AAACCCAAGGTTAAAC-1         0        25         19         1   
1           2  AAACCCACAAATCAAG-1         0         8         18         3   
2           3  AAACCCACAAGTATCC-1         0         8          7         0   
3           4  AAACCCACAGGTTACT-1         0         9         13         2   
4           5  AAACCCACATGATCTG-1         0        14          1        13   

   Hu.CD279  Hu.CD37  Hu.CD99  Hu.LOX.1  ...  RNA.weight  ADT.weight  \
0         0        0        9         0  ...    0.529892    0.470108   
1         0        0       16         0  ...    0.603510    0.396490   
2         1        0        5         1  ...    0.657347    0.342653   
3         0        0       13         2  ...    0.423450    0.576550   
4         3        0        3         1  ...    0.525541    0.474459   

   wsnn_res.2  seurat_clusters  wsnn_res.1  predicted.celltype.score  \
0          10             

In [3]:
num_barcodes = adt_counts["cons_bc"].nunique()

print("number of unique barcodes is : ", num_barcodes)

adt_counts["cons_bc"].head() # length of 20. the subsequent analysis has the length to be subset to 15 and donor id to be added here for analysis

number of unique barcodes is :  3066


0    GTCTCGTTCTGGGTGGTAGT
1                     NaN
2                     NaN
3    CCCTGGTTGTCTGTGTTTGT
4    AGGAGGGACATTCAGGGACT
Name: cons_bc, dtype: object

In [4]:
adt_obj = adt_counts

adt_obj["cons_bc"] = adt_counts["cons_bc"].str[:15]
adt_obj["barcode_donor"] = adt_obj["cons_bc"].str.cat(adt_counts['donor_id'], sep='_')

len(adt_obj["barcode_donor"])
adt_obj["barcode_donor"].head(30)

# no. of barcodes after subsetting and attaching donor - 9768
# lots of NA values. try dropNA ? 

adt_obj["barcode_donor"].isna().sum() # 4259 are NA 
adt_obj = adt_obj[adt_obj["barcode_donor"].notna()]
adt_obj["barcode_donor"].isna().sum()
len(adt_obj["barcode_donor"]) # 5509 barcodes remaining

5509

In [5]:
# subsetting adt_obj to hscs 
print(adt_obj['populations'].head())
adt_obj = adt_obj[adt_obj["populations"] == "HSC"]
len(adt_obj)

# no. left are now 439 

0                           Mk
3                           Mk
4    early_lymphoid_progenitor
6                           Mk
7                  Mk-ery_prog
Name: populations, dtype: object


439

In [6]:
# now should have the same barcodes for fate and X_rna 

fate_obj = pd.read_csv("/vast/projects/Sisseq/givanna/making_sense_of_files/fate_readout.csv")
fate_obj.columns # (barcode x day_PX for X in A,B,C)
fate_obj.head() # we have barcode_first_15_bp 

# tryig to overlap the barcode_donor col of the prev dataset with the bcode_first_15bp of this dataset 

overlap = set(adt_obj["barcode_donor"]) & set(fate_obj["bcode_first_15bp"])

print(f"Number of overlapping barcodes: {len(overlap)}")
print(list(overlap)[:20])

print(fate_obj["bcode_first_15bp"].head())
adt_obj["barcode_donor"].head()

# we are stuck at 112 again. Number of overlapping barcodes: 112

# abandon this and work on the 439 for now. 


Number of overlapping barcodes: 112
['TACAGGCTGTACGTC_P', 'ACCTGCGTGTATGTC_1', 'CTGCAAGGTTGCTAT_U', 'TCGAGATTCATCCTC_1', 'TGGAGTCTCTGGCTC_P', 'AGGTGTTTGTGTCTC_1', 'ATCAGGTTGACCGAG_1', 'TAGAGTTAGAACGAG_1', 'TTCTGAGAGTCGCTG_1', 'TAGCAATCTTTCAAA_U', 'AATGAATCTATGGAG_U', 'CACTCGATGTGTGTC_1', 'CCCAGTGACATAGTG_1', 'GTCCAAGCCACGATT_U', 'CTTGGTACATTGTTT_U', 'GATGGTAGTACGTAC_U', 'CGGTGGTTCTCTCTC_1', 'TTTCTATGAGAAACA_V', 'TTGTCGCTGTGCCTC_P', 'GAGTGGCTCTGTCAG_1']
0    AAACAAACGAGGGAT_U
1    AAACAAACTTGGAAT_U
2    AAACAAAGTAGGCAG_U
3    AAACAAGCGTAGTAG_U
4    AAACAAGGCTCGGAG_U
Name: bcode_first_15bp, dtype: object


28     CTGCCTGGCTTCGTA_1
86     ACCTGGGTGAGWGTG_1
87     TTGACCAAGATTGTG_1
150    TTCTGAGAGTCGCTG_1
151    AACTGACACACAGTG_1
Name: barcode_donor, dtype: object

In [7]:
# reading rna counts scanpy

rna_counts = sc.read_mtx("/vast/projects/Sisseq/givanna/making_sense_of_files/X_rna_counts.mtx") # of type andata 

print(rna_counts)
# just intersect w adt_obj. 

AnnData object with n_obs × n_vars = 9768 × 36601


In [10]:
import anndata as ad
from scipy.sparse import csr_matrix

In [11]:
rna_counts.X

<9768x36601 sparse matrix of type '<class 'numpy.float32'>'
	with 44068873 stored elements in Compressed Sparse Row format>

In [15]:
rna_counts.obsnames = [f"Cell_{i:d}" for i in range(rna_counts.n_obs)]
rna_counts.obs_names[:10]

Index(['0', '1', '2', '3', '4', '5', '6', '7', '8', '9'], dtype='object')

## Step 1: Inspect fate_readout.csv columns

Before we can compute per-barcode dominance, we need to know exactly how the
lineage x day x plate columns are named in `fate_readout.csv`. Run this once,
eyeball the printed column list, then fix `LINEAGE_COLS` in the next cell to
match. We already know from `main.R` that columns are not simply `day_celltype`
— there's also plate (PA/PB/PC) baked in, so we sum across plates per
day/lineage before scoring.

In [ ]:
# fate_obj is already loaded above (cell 6) from fate_readout.csv
# just re-print columns here so this section is self-contained if run standalone
print(fate_obj.shape)
print(fate_obj.columns.tolist())

## Step 2 (superseded)

The automated regex/threshold classifier that used to live here has been removed — its lineage-column auto-mapping was unverified and produced skewed group sizes (cDC1=1, Lymph=11, pDC=8 vs multipotent=606). Use **Step 2b/2c below** (NMF cluster grid + manual labelling) instead, which is what now defines `barcode_labels` for Step 4.

## Step 2b (replaces automated threshold labelling): NMF clustering + manual visual re-labelling

The Step 2 `LINEAGE_COLS` auto-grouping was a regex guess I flagged as unverified
— and the resulting group sizes (cDC1=1, Lymph=11, pDC=8 vs. multipotent=606,
Ery=447) strongly suggest it mis-mapped columns for the rarer lineages. Rather
than debug a threshold rule you can't see behind, this rebuilds the same style
of NMF cluster-trajectory plot as your uploaded image, from `fate_readout.csv`
directly, so you can hand-label clusters the way you did originally.

**First**, print the actual columns and confirm/fix the (lineage, day) parser —
this is the one thing that has to be right before anything downstream is
trustworthy.

In [ ]:
import re

print(fate_obj.columns.tolist())

Edit `LINEAGE_NAMES` / the parser below if the printed columns don't follow a
`<lineage>...<Dxx>...` naming pattern. The parser extracts a day token
(`D7`, `D10`, ...) and a lineage name from each column, then **sums across
plate replicates** (PA/PB/PC) that map to the same (lineage, day) — so plate
never becomes a hidden factor in the clustering.

In [ ]:
LINEAGE_NAMES = ["Ery", "Lymph", "Mast", "Mye", "Rest", "cDC1", "cDC2", "pDC"]
DAY_PATTERN = re.compile(r"(D\d+)", re.IGNORECASE)

def parse_lineage_day(col):
    day_match = DAY_PATTERN.search(col)
    day = day_match.group(1).upper() if day_match else None
    # longest matching lineage name wins (avoids e.g. a hypothetical short name
    # being a substring of a longer one)
    candidates = [lin for lin in LINEAGE_NAMES if re.search(re.escape(lin), col, re.IGNORECASE)]
    lineage = max(candidates, key=len) if candidates else None
    return lineage, day

parsed = {col: parse_lineage_day(col) for col in fate_obj.columns}
unparsed = [c for c, (lin, day) in parsed.items() if lin is None or day is None]

print("Parsed (lineage, day) per column -- SPOT CHECK THIS:")
for col, (lin, day) in parsed.items():
    if lin is not None and day is not None:
        print(f"  {col!r:40s} -> lineage={lin}, day={day}")

print(f"\n{len(unparsed)} columns did NOT parse to a (lineage, day) pair (fine if these are id/plate/patient columns):")
print(unparsed)

In [ ]:
# Build barcode x (lineage, day) matrix, summing plate replicates that share
# the same (lineage, day) pair.
fate_indexed = fate_obj.set_index("bcode_first_15bp")

lineage_day_cols = {}  # (lineage, day) -> list of columns
for col, (lin, day) in parsed.items():
    if lin is not None and day is not None:
        lineage_day_cols.setdefault((lin, day), []).append(col)

days_sorted = sorted({day for (_, day) in lineage_day_cols}, key=lambda d: int(d[1:]))
print("Days found:", days_sorted)

wide = pd.DataFrame(index=fate_indexed.index)
for (lin, day), cols in lineage_day_cols.items():
    wide[(lin, day)] = fate_indexed[cols].clip(lower=0).sum(axis=1)

wide.columns = pd.MultiIndex.from_tuples(wide.columns, names=["lineage", "day"])
wide = wide.reindex(columns=pd.MultiIndex.from_product([LINEAGE_NAMES, days_sorted], names=["lineage", "day"]), fill_value=0)

print(wide.shape, "barcodes x (lineage, day) features")
wide.head()

In [ ]:
from sklearn.decomposition import NMF

X = wide.fillna(0).clip(lower=0).values

N_CLUSTERS = 18  # matches your original clustering count; adjust and re-run to compare

nmf = NMF(n_components=N_CLUSTERS, init="nndsvda", random_state=42, max_iter=1000)
W = nmf.fit_transform(X)   # barcodes x components
H = nmf.components_        # components x (lineage, day) features

cluster_assignment = W.argmax(axis=1)  # hard-assign each barcode to its top component
barcode_cluster_df = pd.DataFrame({
    "barcode": wide.index,
    "cluster": cluster_assignment,
})

print(barcode_cluster_df["cluster"].value_counts().sort_index())

In [ ]:
import matplotlib.pyplot as plt

# reproduce the same grid-of-trajectories plot style as your original
# hand-labelling image: one subplot per cluster, one line per lineage,
# mean value across barcodes in that cluster at each day.
n_clusters_actual = barcode_cluster_df["cluster"].nunique()
n_cols = 4
n_rows = int(np.ceil(n_clusters_actual / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows), sharex=False)
axes = axes.flatten()

for i in range(n_clusters_actual):
    ax = axes[i]
    members = barcode_cluster_df.loc[barcode_cluster_df["cluster"] == i, "barcode"]
    n_bc = len(members)
    for lin in LINEAGE_NAMES:
        vals = wide.loc[members, lin][days_sorted].mean(axis=0)
        ax.plot(days_sorted, vals.values, marker="o", label=lin)
    ax.set_title(f"Cluster {i} has {n_bc} barcodes")

for j in range(n_clusters_actual, len(axes)):
    fig.delaxes(axes[j])

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper right")
plt.tight_layout()
plt.show()

## Step 2c: Enter your manual cluster labels here

Look at the grid above the way you looked at your original image, and fill in
`CLUSTER_LABELS` below: cluster id -> `"multipotent"` or the dominant lineage
name. Any cluster you leave out defaults to `"unassigned"` and its barcodes
will be dropped before the MOFA step.

In [ ]:
# ==== EDIT THIS after inspecting the cluster grid above ====
CLUSTER_LABELS = {
    # 0: "multipotent",
    # 1: "Ery",
    # ...
}

barcode_cluster_df["group_label"] = barcode_cluster_df["cluster"].map(CLUSTER_LABELS).fillna("unassigned")
barcode_labels = barcode_cluster_df.set_index("barcode")[["group_label"]]

print(barcode_labels["group_label"].value_counts(dropna=False))

## Step 3: Preprocess ADT and RNA

- **ADT**: CLR (centered log-ratio) normalisation is the standard for CITE-seq
  protein counts (rather than plain log1p) since it handles the compositional,
  low-dimensionality nature of ADT panels better. Done per-cell, across
  proteins.
- **RNA**: standard scanpy flow — CPM-normalise (`target_sum=1e4` counts-per-
  10k, or `1e6` for true CPM), log1p, then subset to highly variable genes
  (HVGs) so MOFA isn't swamped by noise genes. Then scale (zero mean, unit
  variance, clipped) since MOFA assumes roughly Gaussian-like continuous
  inputs per view.

In [ ]:
# ---- ADT: CLR normalisation ----
adt_feature_cols = [c for c in adt_counts.columns if c.startswith(("Hu.", "HuMs", "Isotype"))]

def clr_normalize(mat):
    """Centered log-ratio, per cell (row), standard for CITE-seq ADT counts."""
    mat = mat.astype(float)
    mat_pseudo = mat + 1  # pseudocount to handle zeros
    log_mat = np.log(mat_pseudo)
    geometric_mean = log_mat.mean(axis=1)
    return log_mat.sub(geometric_mean, axis=0)

adt_clr = clr_normalize(adt_counts[adt_feature_cols])
adt_clr["cell"] = adt_counts["cell"].values
adt_clr["cons_bc"] = adt_counts["cons_bc"].str[:15].values
adt_clr["donor_id"] = adt_counts["donor_id"].values
adt_clr["barcode_donor"] = adt_clr["cons_bc"].str.cat(adt_clr["donor_id"], sep="_")
adt_clr["populations"] = adt_counts["populations"].values

adt_clr.head()

In [ ]:
# ---- RNA: CPM normalise, log1p, HVG subset, scale ----
# rna_counts loaded above via sc.read_mtx; rows = cells (as exported by
# writeMM(t(...)) in the R script), cols = genes.

import os

# NOTE: the R export script (main.R) has a bug -- the writeMM() call for the
# .mtx uses file.path(out_dir, ...) but the two writeLines() calls right after
# it don't, so gene_names/cell_ids can land one directory up from the .mtx
# (in the R working directory at time of export) rather than alongside it.
# Check both candidate locations and use whichever actually matches the
# gene/cell counts in rna_counts, rather than assuming a fixed path.
CANDIDATE_DIRS = [
    "/vast/projects/Sisseq/givanna/making_sense_of_files",
    "/vast/projects/Sisseq/givanna/code",
    "/vast/projects/Sisseq/givanna",
]

def load_names_checked(filename, expected_len):
    for d in CANDIDATE_DIRS:
        path = os.path.join(d, filename)
        if not os.path.exists(path):
            continue
        names = open(path).read().splitlines()
        if len(names) == expected_len:
            print(f"OK: {path} has {len(names)} lines, matches expected {expected_len}")
            return names
        else:
            print(f"SKIP: {path} has {len(names)} lines, expected {expected_len} -- stale/wrong file, not using it")
    raise FileNotFoundError(
        f"Could not find a {filename} with {expected_len} lines in any of {CANDIDATE_DIRS}. "
        f"Likely the R export never wrote a correct copy at any known path -- re-run the export "
        f"with writeLines(..., file.path(out_dir, '{filename}')) (missing file.path() is the bug) "
        f"and point CANDIDATE_DIRS at wherever it lands."
    )

gene_names = load_names_checked("X_rna_counts_gene_names.txt", rna_counts.n_vars)
cell_ids = load_names_checked("X_rna_counts_cell_ids.txt", rna_counts.n_obs)

rna_counts.var_names = gene_names
rna_counts.obs_names = cell_ids
rna_counts.var_names_make_unique()

sc.pp.normalize_total(rna_counts, target_sum=1e4)
sc.pp.log1p(rna_counts)

sc.pp.highly_variable_genes(rna_counts, n_top_genes=2000, flavor="seurat")
rna_hvg = rna_counts[:, rna_counts.var["highly_variable"]].copy()

sc.pp.scale(rna_hvg, max_value=10)

rna_hvg

## Step 4: Broadcast barcode-level group label onto every daughter cell

Every profiled cell carries `cons_bc` + `donor_id` in its metadata. We map
each cell's `barcode_donor` key through `barcode_labels["group_label"]` and
drop cells whose barcode has no fate readout at all (can't be assigned to any
group).

In [ ]:
group_map = barcode_labels["group_label"].to_dict()

adt_clr["group"] = adt_clr["barcode_donor"].map(group_map)

n_before = len(adt_clr)
adt_clr = adt_clr[adt_clr["group"].notna()].copy()
n_after = len(adt_clr)
print(f"ADT cells: {n_before} -> {n_after} after dropping cells with no fate-matched barcode")

print(adt_clr["group"].value_counts())

## Step 5: Build the multi-view, multi-group MOFA input and run

Two views (RNA, ADT), groups = the categorical `group` label (dominant
lineage name, or `"multipotent"`) from Step 4. Cells that don't overlap
between RNA and ADT (matched on `cell` id) are dropped — MOFA needs the same
sample set present across views within a group (missing entries are allowed
and handled internally, but we still need consistent sample IDs).

This uses `mofapy2`'s `entry_point` API directly so we can pass `groups`
explicitly as long-format data, which is the simplest way to hand it
per-cell group assignments.

In [ ]:
# pip install mofapy2 --break-system-packages   # if not already installed
from mofapy2.run.entry_point import entry_point

# align RNA (anndata, obs_names = cell ids) with ADT (dataframe, 'cell' col)
rna_df = pd.DataFrame(
    rna_hvg.X if not hasattr(rna_hvg.X, "toarray") else rna_hvg.X.toarray(),
    index=rna_hvg.obs_names,
    columns=rna_hvg.var_names,
)

common_cells = sorted(set(rna_df.index) & set(adt_clr["cell"]))
print(f"Cells common to RNA and ADT (post group-filter): {len(common_cells)}")

rna_df = rna_df.loc[common_cells]
adt_indexed = adt_clr.set_index("cell").loc[common_cells]
cell_groups = adt_indexed["group"]  # one group label per cell, aligned to common_cells

# ---- build long-format tables mofapy2 expects: sample, group, feature, value, view ----
def to_long(df, view_name, groups):
    long = df.stack().reset_index()
    long.columns = ["sample", "feature", "value"]
    long["view"] = view_name
    long["group"] = long["sample"].map(groups)
    return long

rna_long = to_long(rna_df, "RNA", cell_groups)
adt_long = to_long(adt_indexed[adt_feature_cols], "ADT", cell_groups)

long_data = pd.concat([rna_long, adt_long], ignore_index=True)
long_data.head()

In [ ]:
ent = entry_point()

ent.set_data_options(
    scale_groups=False,
    scale_views=True,   # important with RNA (2000 features) + ADT (~180) at very different scales
)

ent.set_data_df(long_data)

ent.set_model_options(
    factors=15,
    spikeslab_weights=True,
    ard_weights=True,
)

ent.set_train_options(
    iter=1000,
    convergence_mode="fast",
    dropR2=0.001,
    gpu_mode=False,
    seed=42,
)

ent.build()
ent.run()

ent.save("/home/claude/mofa_unipotency_groups.hdf5")

## Step 6: Variance explained per factor per group

This is the actual check for whether the grouping did anything: if certain
factors explain variance strongly in one group (e.g. `Ery`) but not others,
those factors are candidates for what distinguishes that lineage's fate
commitment from multipotent/other cells. Flat variance-explained across all
groups for every factor would mean the grouping isn't picking up meaningful
structure — in which case, per the original plan, fall back to running
everything ungrouped (`groups=False`, single group) as a baseline comparison.

In [ ]:
import mofax as mfx

model = mfx.mofa_model("/home/claude/mofa_unipotency_groups.hdf5")

r2 = model.get_r2()  # columns typically: Factor, View, Group, R2
print(r2.head(20))

r2_pivot = r2.pivot_table(index="Factor", columns=["Group", "View"], values="R2")
r2_pivot

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(10, max(4, 0.4 * len(r2_pivot))))
sns.heatmap(r2_pivot, cmap="viridis", ax=ax, cbar_kws={"label": "R2 (variance explained)"})
ax.set_title("Variance explained per factor, per group x view")
plt.tight_layout()
plt.show()

# quick read: for each factor, which group(s) have disproportionately high R2?
# those are your "this factor is doing something specific to this fate" candidates.
model.close()